# 03 - Evaluation, threshold và Top-p

## Ngưỡng và ngân sách kiểm tra
Ngưỡng chính giảm `(20×FN + FP)/N_validation`. Tỷ lệ 20:1 là giả định học thuật. Có xét cả lựa chọn không phát cảnh báo; khi hòa chi phí, ưu tiên recall cao. Quyết định và hash model được ghi vào `decision_lock.json` trước khi đọc test. Ngưỡng F1 là đối chiếu đã chọn trên validation. Top-p dùng `k=ceil(p×N_test)`; Precision@k=TP/k, Recall@k=TP/tổng fraud, Lift=Precision@k/tỷ lệ fraud. Tăng k thường tìm thêm fraud nhưng tăng khối lượng kiểm tra.

Notebook chọn threshold trên validation rồi mới đánh giá test một lần. Threshold chính tối thiểu hóa chi phí học thuật `20 x FN + 1 x FP`; threshold tối đa F1 được báo cáo để đối chiếu.

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'config' / 'a1_config.json').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from IPython.display import Image, display
import pandas as pd
from src.evaluation import evaluate_project

summary = evaluate_project(PROJECT_ROOT, bootstrap_iterations=1000)
print(summary['selection_reason'])
print('Selected:', summary['selected_family'])
print('Test AP:', summary['test']['average_precision'])

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / 'outputs/tables/model_comparison.csv')
top_p = pd.read_csv(PROJECT_ROOT / 'outputs/tables/top_p_metrics.csv')
errors = pd.read_csv(PROJECT_ROOT / 'outputs/tables/error_examples.csv')
display(comparison)
display(top_p)
display(errors)

In [ ]:
for name in ['validation_pr_curve.png', 'test_pr_curve.png', 'test_confusion_matrix.png', 'top_p_performance.png', 'seaborn_error_cases_multivariate.png', 'feature_importance.png']:
    display(Image(filename=str(PROJECT_ROOT / 'outputs/figures' / name)))

Hình Seaborn FP/FN mô tả 10 cảnh báo nhầm và 10 mẫu bỏ sót được chọn sau khi khóa threshold. Nó cho thấy LogAmount, V14 và score ở các lỗi được chọn; không suy ra nguyên nhân gian lận hay hành vi khách hàng từ biến ẩn danh.

## Diễn giải kết quả của lần chạy
Các con số dưới đây được tính từ output vừa tạo.

In [ ]:
display(pd.DataFrame([summary['test']['cost_threshold_metrics'], summary['test']['validation_f1_threshold_metrics']], index=['Ngưỡng chi phí', 'Ngưỡng F1 validation']))
for row in top_p.itertuples():
    print(f'Top {row.top_p_percent:g}%: kiểm tra {row.k} dòng, tìm {row.tp}/{summary["test"]["fraud"]} fraud; ' f'Precision={row.precision_at_k:.2%}, Recall={row.recall_at_k:.2%}, Lift={row.lift_at_k:.2f}.')